# N-CMAPSS Preprocessing — Phase 2

Turns `loaders.py`'s already-labeled, per-file `.npz` output (see
`deliverables/pipeline_run_record.docx` and
`deliverables/data_engineering_baseline.docx`) into data the MA1DCNN can
actually train on: per-sensor-channel normalization, optional noise
augmentation, and a time-ordered train/validation/test split.

## What this notebook does, in order

1. **Reads** loaders.py's saved `.npz` files, one at a time (never all 9
   at once) -- the same chunked discipline `loaders.py` uses, applied
   here to a second stage of the pipeline instead of being reinvented.
2. **Splits** each file's rows into train/validation by WHOLE engine
   unit, never by individual row -- explained in Section 4 below.
3. **Computes** per-channel mean/std from TRAINING rows only, streamed
   across all 9 files without ever holding more than one file in memory.
4. **Normalizes** every channel to zero mean / unit variance using those
   training-only statistics.
5. **Augments** the training split (only) with Gaussian and impulse
   noise, per this module's original design note citing Liao et al.
   (2024)'s synthetic-to-real-gap precedent.
6. **Saves** train/validation `.npz` files per input file (plus a
   normalized test file, if `loaders.py` has produced one).

Every function below has its own test cell directly after it, using
small synthetic data -- no Google Drive or real dataset needed to run
and understand any single piece of this notebook. Only the final "Run
it" cell needs the real, Drive-mounted data.

In [1]:
import os
import tempfile

import numpy as np

## 1. Constants

Every judgment call in this notebook -- what counts as a channel to
normalize, how big the validation split is, how harsh the training
noise is -- is named here, once, rather than buried inside a function.

In [2]:
# The 9 files loaders.py already processed and saved. Same fixed list
# and order as loaders.py's NCMAPSS_FILES, so the two modules never
# disagree about what "all 9 files" means.
NCMAPSS_FILES = [
    "N-CMAPSS_DS01-005.h5",
    "N-CMAPSS_DS02-006.h5",
    "N-CMAPSS_DS03-012.h5",
    "N-CMAPSS_DS04.h5",
    "N-CMAPSS_DS05.h5",
    "N-CMAPSS_DS06.h5",
    "N-CMAPSS_DS07.h5",
    "N-CMAPSS_DS08a-009.h5",
    "N-CMAPSS_DS08c-008.h5",
]

# Where loaders.py already saved its labeled, per-file .npz output.
# This notebook only ever READS from here, never writes here.
DEFAULT_PROCESSED_DIR = "/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/processed"

# Where THIS notebook's own output (normalized, split, optionally
# noise-augmented) gets saved -- a separate folder, so loaders.py's
# output is never at risk of being overwritten.
DEFAULT_PREPROCESSED_DIR = "/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/preprocessed"

# Which groups get z-score normalized. A (unit/cycle/Fc/hs) is
# bookkeeping, Y (RUL) is only ever used to build the label, and labels
# are already binary 0/1 -- normalizing any of those three would not
# make sense.
NORMALIZE_GROUPS = ["W", "X_s", "X_v"]

# Column layout of the A group, per docs/data-dictionary.md: column 0 is
# the engine unit ID -- what the train/validation split below groups by.
A_UNIT_COLUMN = 0

# Fraction of each file's DEV units held out for validation. This is a
# judgment call (20% is a common default), not a value that comes from
# Akindoju's thesis or the N-CMAPSS papers -- documented here so it is
# easy to find and revise.
VALIDATION_UNIT_FRACTION = 0.2

# Training-time noise-augmentation level, in dB SNR. -6dB is the
# synthetic-to-real-gap precedent cited in this module's original design
# note (Liao et al., 2024). It is intentionally much harsher than the
# 0/10/20dB levels docs/praxis/04-hypotheses.md (H1) later tests the
# TRAINED model against -- augmentation noise should be at least as
# harsh as evaluation noise, or the model never actually learns to
# handle it.
AUGMENT_GAUSSIAN_SNR_DB = -6.0

# What fraction of individual sensor readings get replaced with an
# impulse (sensor-glitch) spike during training augmentation, and how
# large that spike is, in units of that channel's own standard
# deviation. Both are judgment calls -- there is no cited source for
# these two specific numbers.
AUGMENT_IMPULSE_FRACTION = 0.01
AUGMENT_IMPULSE_MAGNITUDE = 3.0

## 2. Mount Google Drive

Identical in behavior to `loaders.py`'s own `mount_google_drive()` --
duplicated here so this notebook can run standalone in a fresh Colab
session.

In [3]:
def mount_google_drive():
    """Mount Google Drive inside a Colab runtime."""
    try:
        from google.colab import drive
    except ImportError:
        print("Not running in Colab -- skipping Drive mount.")
        return

    print("Mounting Google Drive ...")
    drive.mount("/content/drive")
    print("Drive mounted.")


mount_google_drive()

Mounting Google Drive ...
Mounted at /content/drive
Drive mounted.


## 3. Reading loaders.py's saved output

`load_processed_file` and `iter_processed_files` are identical in
behavior to `loaders.py`'s own functions of the same name. Every later
section in this notebook reads through these two functions, never
through raw `np.load` calls scattered around -- so there is exactly one
place that knows the on-disk file-naming convention.

In [4]:
def load_processed_file(filename, out_dir=DEFAULT_PROCESSED_DIR, split="dev"):
    """Load ONE of loaders.py's already-saved, labeled .npz files."""
    out_name = filename.replace(".h5", f"_{split}.npz")
    out_path = os.path.join(out_dir, out_name)
    with np.load(out_path) as npz:
        return {key: npz[key] for key in npz.files}


def iter_processed_files(out_dir=DEFAULT_PROCESSED_DIR, split="dev", files=None):
    """Yield one processed file's data at a time -- never all 9 in memory at once."""
    files = files if files is not None else NCMAPSS_FILES
    for filename in files:
        yield filename, load_processed_file(filename, out_dir=out_dir, split=split)

In [5]:
def _test_load_processed_file():
    with tempfile.TemporaryDirectory() as out_dir:
        fake = {"W": np.array([[1.0, 2.0]]), "labels": np.array([[0, 0, 0, 0, 0]], dtype=np.int8)}
        # load_processed_file expects a "<name>_{split}.npz" file -- save
        # directly under that convention rather than the plain .h5 name.
        np.savez_compressed(os.path.join(out_dir, "FAKE_dev.npz"), **fake)

        reloaded = load_processed_file("FAKE.h5", out_dir=out_dir, split="dev")
        assert np.array_equal(reloaded["W"], fake["W"])
        assert np.array_equal(reloaded["labels"], fake["labels"])

    print("load_processed_file: PASSED")


_test_load_processed_file()

load_processed_file: PASSED


## 4. Time-ordered train/validation split, per engine unit

Splits each file's rows into train and validation by WHOLE engine unit,
never by individual row.

**Why split by unit, not by row.** Consecutive rows within one unit's
run-to-failure trajectory are highly correlated -- they are the same
simulated aircraft, minutes apart. Splitting individual rows randomly
between train and validation would let the model see one cycle of a
unit in training and the very next cycle of that SAME unit in
validation: an easy, unrealistic shortcut that would make validation
accuracy look far better than real generalization. Keeping every unit's
rows entirely in one split avoids that leakage.

**Why "time-ordered" here means "never row-shuffled," not "units sorted
by calendar time."** Per `docs/data-dictionary.md`, unit numbering
restarts per file and carries no cross-unit calendar meaning -- unit 3
in one file did not necessarily fly before or after unit 5 in that same
file. What matters for correctness is that each unit's own cycle order
is never shuffled or split across sets, which assigning whole units (in
a fixed, deterministic order) guarantees.

In [6]:
def split_units_train_val(unit_ids, val_fraction=VALIDATION_UNIT_FRACTION):
    """Split one file's rows into train/validation by WHOLE engine unit.

    Parameters
    ----------
    unit_ids : np.ndarray, shape (n_rows,)
        The A[:, 0] column for one file -- which unit each row belongs to.
    val_fraction : float
        Fraction of this file's DISTINCT units (not rows) held out.

    Returns
    -------
    train_mask, val_mask : np.ndarray[bool], shape (n_rows,)
        Row-aligned boolean masks. Every row is in exactly one of the two.
    train_units, val_units : np.ndarray
        The distinct unit IDs assigned to each split, for logging/checks.
    """
    unique_units = np.unique(unit_ids)  # ascending, deduplicated

    if len(unique_units) < 2:
        # Too few units to hold any out -- everything goes to train
        # rather than raising an error, so this still works on tiny
        # synthetic test fixtures with a single unit.
        print(f"    only {len(unique_units)} unit(s) present -- skipping validation split")
        train_mask = np.ones(unit_ids.shape[0], dtype=bool)
        val_mask = np.zeros(unit_ids.shape[0], dtype=bool)
        return train_mask, val_mask, unique_units, unique_units[:0]

    n_val = max(1, round(val_fraction * len(unique_units)))
    val_units = unique_units[-n_val:]     # deterministic: always the LAST n_val unit IDs
    train_units = unique_units[:-n_val]   # everything else

    train_mask = np.isin(unit_ids, train_units)
    val_mask = np.isin(unit_ids, val_units)

    return train_mask, val_mask, train_units, val_units

In [7]:
def _test_split_units_train_val():
    # 5 units, 10 rows each, in order -- a stand-in for one file's A[:, 0].
    unit_ids = np.repeat([1, 2, 3, 4, 5], 10)

    train_mask, val_mask, train_units, val_units = split_units_train_val(unit_ids, val_fraction=0.2)

    # 20% of 5 units = 1 unit held out -- must be unit 5, the last one.
    assert val_units.tolist() == [5], val_units
    assert train_units.tolist() == [1, 2, 3, 4], train_units

    # Every row is in exactly one split -- never both, never neither.
    assert (train_mask | val_mask).all()
    assert not (train_mask & val_mask).any()
    assert train_mask.sum() == 40
    assert val_mask.sum() == 10

    # Edge case: a single-unit file should not crash, and should keep
    # everything in train rather than holding out its only unit.
    single_unit_ids = np.repeat([7], 15)
    tm, vm, tu, vu = split_units_train_val(single_unit_ids, val_fraction=0.2)
    assert tm.all() and not vm.any()
    assert len(vu) == 0

    print("split_units_train_val: PASSED")


_test_split_units_train_val()

    only 1 unit(s) present -- skipping validation split
split_units_train_val: PASSED


## 5. Streaming per-channel normalization statistics (training rows only)

Computes per-channel mean/std across ALL files' TRAINING rows only,
streaming one file at a time -- the same running-total trick that keeps
`loaders.py`'s memory flat, applied here to statistics instead of raw
rows, using the single-pass variance identity
`Var(X) = E[X^2] - E[X]^2` so a second pass over the data is never
needed.

**Why training rows only.** If validation or test rows leaked into
these statistics, the model would be implicitly "peeking" at data it is
later evaluated on, inflating validation/test performance in a way that
would not hold up on genuinely new data. The test cell below checks
this directly, not just the arithmetic.

In [8]:
def compute_channel_stats(out_dir=DEFAULT_PROCESSED_DIR, files=None,
                           groups=NORMALIZE_GROUPS, val_fraction=VALIDATION_UNIT_FRACTION):
    """Compute per-channel mean/std across ALL files' TRAINING rows only.

    Returns
    -------
    dict[str, dict[str, np.ndarray]]
        e.g. stats["W"]["mean"], stats["W"]["std"] -- one mean/std
        vector per channel within each group.
    """
    files = files if files is not None else NCMAPSS_FILES

    # Running totals per group -- small vectors (one value per channel),
    # never anything the size of the dataset itself.
    count = 0
    sums = {g: None for g in groups}
    sums_sq = {g: None for g in groups}

    for i, filename in enumerate(files, start=1):
        print(f"[{i}/{len(files)}] Accumulating stats from {filename} ...")
        data = load_processed_file(filename, out_dir=out_dir)  # one file's arrays in memory

        unit_ids = data["A"][:, A_UNIT_COLUMN]
        train_mask, _, train_units, val_units = split_units_train_val(unit_ids, val_fraction)
        print(f"    {len(train_units)} train unit(s), {len(val_units)} validation unit(s), "
              f"{int(train_mask.sum()):,} train rows")

        for g in groups:
            train_rows = data[g][train_mask]  # only this file's TRAIN rows contribute
            if sums[g] is None:
                sums[g] = train_rows.sum(axis=0)
                sums_sq[g] = (train_rows ** 2).sum(axis=0)
            else:
                sums[g] += train_rows.sum(axis=0)
                sums_sq[g] += (train_rows ** 2).sum(axis=0)

        count += int(train_mask.sum())
        del data  # free this file's arrays before loading the next one

    stats = {}
    for g in groups:
        mean = sums[g] / count
        var = sums_sq[g] / count - mean ** 2         # single-pass variance identity
        std = np.sqrt(np.maximum(var, 0))             # clip tiny negative values from floating-point rounding
        std = np.where(std < 1e-8, 1.0, std)           # a ~zero-variance channel would otherwise divide by ~zero
        stats[g] = {"mean": mean, "std": std}
        print(f"    {g}: mean={np.round(mean, 3)}  std={np.round(std, 3)}")

    print(f"Done -- stats computed from {count:,} total training rows across {len(files)} files")
    return stats

In [9]:
def _test_compute_channel_stats():
    with tempfile.TemporaryDirectory() as out_dir:
        rng = np.random.default_rng(7)
        # 2 units, 500 rows each. val_fraction=0.5 holds out unit 2
        # entirely, so only unit 1's rows should count toward stats.
        n_per_unit = 500
        unit_col = np.repeat([1, 2], n_per_unit)
        n = len(unit_col)

        # Known distribution for unit 1's W column: mean=100, std=5.
        # Unit 2 is given a wildly different distribution specifically
        # so a leakage bug would be impossible to miss.
        w = np.zeros((n, 2))
        w[unit_col == 1] = rng.normal(loc=100.0, scale=5.0, size=(n_per_unit, 2))
        w[unit_col == 2] = rng.normal(loc=-999.0, scale=1.0, size=(n_per_unit, 2))

        data = {
            "W": w,
            "X_s": rng.random((n, 3)),
            "X_v": rng.random((n, 3)),
            "A": np.stack([unit_col, np.zeros(n), np.zeros(n), np.ones(n)], axis=1),
            "Y": rng.integers(0, 300, size=(n, 1)),
            "labels": np.zeros((n, 5), dtype=np.int8),
        }
        np.savez_compressed(os.path.join(out_dir, "FAKE_dev.npz"), **data)

        stats = compute_channel_stats(out_dir=out_dir, files=["FAKE.h5"], groups=["W"], val_fraction=0.5)

        # If unit 2's rows leaked into the stats, this mean would be
        # nowhere near 100 -- this IS the leakage check, not just an
        # arithmetic check.
        assert np.allclose(stats["W"]["mean"], 100.0, atol=1.0), stats["W"]["mean"]
        assert np.allclose(stats["W"]["std"], 5.0, atol=1.0), stats["W"]["std"]

    print("compute_channel_stats: PASSED (confirmed validation-unit rows are excluded)")


_test_compute_channel_stats()

[1/1] Accumulating stats from FAKE.h5 ...
    1 train unit(s), 1 validation unit(s), 500 train rows
    W: mean=[99.94  99.337]  std=[4.686 4.708]
Done -- stats computed from 500 total training rows across 1 files
compute_channel_stats: PASSED (confirmed validation-unit rows are excluded)


## 6. Applying normalization

`(x - mean) / std`, per channel, using the training-derived statistics
from Section 5. `A`, `Y`, and `labels` pass through completely
unchanged.

In [10]:
def apply_normalization(data, stats, groups=NORMALIZE_GROUPS):
    """Return a NEW dict with `groups` z-score normalized; everything else untouched."""
    out = dict(data)  # shallow copy -- untouched keys share the same array object, which is fine since we never mutate them
    for g in groups:
        out[g] = (data[g] - stats[g]["mean"]) / stats[g]["std"]
    return out

In [11]:
def _test_apply_normalization():
    rng = np.random.default_rng(0)
    w = rng.normal(loc=50.0, scale=10.0, size=(1000, 4))  # known mean=50, std=10
    x_s = rng.random((1000, 14))
    data = {
        "W": w, "X_s": x_s, "X_v": rng.random((1000, 14)),
        "A": rng.integers(0, 5, size=(1000, 4)),
        "Y": rng.integers(0, 300, size=(1000, 1)),
        "labels": np.zeros((1000, 5), dtype=np.int8),
    }
    stats = {
        "W": {"mean": w.mean(axis=0), "std": w.std(axis=0)},
        "X_s": {"mean": x_s.mean(axis=0), "std": x_s.std(axis=0)},
        "X_v": {"mean": data["X_v"].mean(axis=0), "std": data["X_v"].std(axis=0)},
    }

    normalized = apply_normalization(data, stats)

    # Normalized channels should now have ~0 mean, ~1 std.
    assert np.allclose(normalized["W"].mean(axis=0), 0, atol=1e-6)
    assert np.allclose(normalized["W"].std(axis=0), 1, atol=1e-6)

    # Untouched groups must be byte-for-byte identical to the input.
    assert np.array_equal(normalized["A"], data["A"])
    assert np.array_equal(normalized["Y"], data["Y"])
    assert np.array_equal(normalized["labels"], data["labels"])

    print("apply_normalization: PASSED")


_test_apply_normalization()

apply_normalization: PASSED


## 7. Noise augmentation for training data (Gaussian + impulse)

Two independent kinds of synthetic sensor noise, applied only to the
TRAINING split later in Section 8 -- validation and test data stay
clean, since they are meant to measure performance on realistic,
unaugmented data.

- **Gaussian noise** models smooth sensor drift, added at a target
  signal-to-noise ratio (SNR) in dB.
- **Impulse noise** models a sudden sensor glitch: a single reading
  jumps by several standard deviations, in a random direction.

In [12]:
def add_gaussian_noise(x, snr_db=AUGMENT_GAUSSIAN_SNR_DB, rng=None):
    """Add zero-mean Gaussian noise to every channel at a target SNR (dB).

    SNR_dB = 10 * log10(signal_power / noise_power), rearranged to solve
    for noise_power:
        noise_power = signal_power / 10^(SNR_dB / 10)

    Signal power is measured PER CHANNEL (per column), so a channel with
    a larger natural scale gets proportionally more noise -- this keeps
    the requested SNR accurate on every channel, not just on average
    across all of them.
    """
    rng = rng or np.random.default_rng()
    signal_power = np.mean(x ** 2, axis=0)             # shape: (n_channels,)
    noise_power = signal_power / (10 ** (snr_db / 10))  # rearranged SNR formula, solved for noise power
    sigma = np.sqrt(noise_power)
    noise = rng.normal(loc=0.0, scale=sigma, size=x.shape)
    return x + noise


def add_impulse_noise(x, fraction=AUGMENT_IMPULSE_FRACTION,
                       magnitude=AUGMENT_IMPULSE_MAGNITUDE, rng=None):
    """Corrupt a random fraction of individual readings with a large spike.

    Returns
    -------
    x_out : np.ndarray, same shape as x
    corrupted_mask : np.ndarray[bool], same shape as x
        True at every position that got a spike -- returned mainly so
        tests (and curious readers) can check the actual corruption rate.
    """
    rng = rng or np.random.default_rng()
    channel_std = x.std(axis=0)  # per-channel scale, so a spike is meaningful relative to that channel
    corrupted_mask = rng.random(x.shape) < fraction
    spike_sign = rng.choice([-1.0, 1.0], size=x.shape)
    spikes = spike_sign * magnitude * channel_std  # broadcasts (n_channels,) across all rows
    x_out = np.where(corrupted_mask, x + spikes, x)
    return x_out, corrupted_mask

In [13]:
def _test_noise_augmentation():
    rng = np.random.default_rng(1)
    x = rng.normal(loc=0.0, scale=1.0, size=(200_000, 3))  # large N so measured SNR is stable

    target_snr_db = 10.0
    noisy = add_gaussian_noise(x, snr_db=target_snr_db, rng=rng)
    actual_noise = noisy - x
    measured_snr_db = 10 * np.log10(np.mean(x ** 2, axis=0) / np.mean(actual_noise ** 2, axis=0))
    # Random sampling means this will not be exact -- within 0.5dB is a
    # tight, reliable tolerance at 200,000 samples.
    assert np.allclose(measured_snr_db, target_snr_db, atol=0.5), measured_snr_db

    x2 = rng.normal(size=(100_000, 2))
    noisy2, mask = add_impulse_noise(x2, fraction=0.05, rng=rng)
    measured_fraction = mask.mean()
    assert abs(measured_fraction - 0.05) < 0.01, measured_fraction
    assert np.all(noisy2[mask] != x2[mask])       # every masked position must actually differ
    assert np.array_equal(noisy2[~mask], x2[~mask])  # every unmasked position must be untouched

    print(f"add_gaussian_noise: PASSED (measured SNR ~{measured_snr_db.mean():.2f}dB vs target {target_snr_db}dB)")
    print(f"add_impulse_noise: PASSED (measured corruption rate {measured_fraction:.3f} vs target 0.05)")


_test_noise_augmentation()

add_gaussian_noise: PASSED (measured SNR ~10.01dB vs target 10.0dB)
add_impulse_noise: PASSED (measured corruption rate 0.051 vs target 0.05)


## 8. Putting it together: process_and_save_preprocessed()

Two passes over the files, exactly like `loaders.py`'s own chunked
design -- never holding more than one file's arrays in memory:

- **Pass 1** streams every file once, computes each file's train/
  validation unit split, and accumulates per-channel normalization
  stats from TRAINING rows only (`compute_channel_stats`).
- **Pass 2** streams every file again, applies the now-known stats to
  normalize `W`/`X_s`/`X_v`, adds noise augmentation to the TRAINING
  rows only, and saves separate train/validation `.npz` files per input
  file.

If a matching TEST-split file also exists on disk (i.e. `loaders.py` has
already been run with `split="test"`), it is normalized with the same
training-derived stats and saved as-is -- no further splitting, no
augmentation, since it represents genuinely held-out data. As of this
praxis's current state, `loaders.py` has only been run with
`split="dev"` (see `deliverables/data_engineering_baseline.docx`,
Section 7), so this branch is expected to be skipped for now -- that is
normal, not an error.

In [14]:
def process_and_save_preprocessed(processed_dir=DEFAULT_PROCESSED_DIR,
                                   out_dir=DEFAULT_PREPROCESSED_DIR,
                                   files=None,
                                   val_fraction=VALIDATION_UNIT_FRACTION,
                                   augment=True):
    """Turn loaders.py's saved dev-split output into train/val/test-ready data."""
    files = files if files is not None else NCMAPSS_FILES
    os.makedirs(out_dir, exist_ok=True)

    print("=== Pass 1: computing per-channel normalization stats (train rows only) ===")
    stats = compute_channel_stats(out_dir=processed_dir, files=files, val_fraction=val_fraction)

    print()
    print("=== Pass 2: normalizing, splitting, saving ===")
    rng = np.random.default_rng(0)  # fixed seed -- augmented noise is reproducible run to run
    summaries = []

    for i, filename in enumerate(files, start=1):
        print(f"[{i}/{len(files)}] Processing {filename} ...")
        data = load_processed_file(filename, out_dir=processed_dir)  # one file's arrays in memory

        unit_ids = data["A"][:, A_UNIT_COLUMN]
        train_mask, val_mask, train_units, val_units = split_units_train_val(unit_ids, val_fraction)

        normalized = apply_normalization(data, stats)
        train_data = {k: v[train_mask] for k, v in normalized.items()}
        val_data = {k: v[val_mask] for k, v in normalized.items()}

        if augment:
            train_data["W"] = add_gaussian_noise(train_data["W"], rng=rng)
            train_data["X_s"], impulse_mask = add_impulse_noise(train_data["X_s"], rng=rng)
            print(f"    augmented {int(train_mask.sum()):,} train rows "
                  f"({AUGMENT_GAUSSIAN_SNR_DB}dB Gaussian on W, "
                  f"{impulse_mask.mean() * 100:.2f}% impulse on X_s)")

        base_name = filename.replace(".h5", "")
        train_path = os.path.join(out_dir, f"{base_name}_train.npz")
        val_path = os.path.join(out_dir, f"{base_name}_val.npz")
        np.savez_compressed(train_path, **train_data)
        np.savez_compressed(val_path, **val_data)
        print(f"    saved {int(train_mask.sum()):,} train rows -> {train_path}")
        print(f"    saved {int(val_mask.sum()):,} val rows   -> {val_path}")

        summary = {
            "filename": filename,
            "n_train_rows": int(train_mask.sum()),
            "n_val_rows": int(val_mask.sum()),
            "train_path": train_path,
            "val_path": val_path,
        }

        # Test-split file is OPTIONAL, per the note above -- skipped
        # gracefully, not an error, if it does not exist yet.
        test_name = filename.replace(".h5", "_test.npz")
        test_in_path = os.path.join(processed_dir, test_name)
        if os.path.exists(test_in_path):
            with np.load(test_in_path) as npz:
                test_data = {k: npz[k] for k in npz.files}
            test_normalized = apply_normalization(test_data, stats)  # same train-derived stats, no augmentation
            test_out_path = os.path.join(out_dir, f"{base_name}_test.npz")
            np.savez_compressed(test_out_path, **test_normalized)
            print(f"    saved {test_data['W'].shape[0]:,} test rows  -> {test_out_path}")
            summary["n_test_rows"] = test_data["W"].shape[0]
            summary["test_path"] = test_out_path
        else:
            print(f"    no test-split file found at {test_in_path} -- skipping (dev-only, so far)")

        summaries.append(summary)
        del data, normalized, train_data, val_data  # free this file's memory before the next iteration

    total_train = sum(s["n_train_rows"] for s in summaries)
    total_val = sum(s["n_val_rows"] for s in summaries)
    print()
    print(f"Done -- {len(files)} files processed, {total_train:,} train rows / {total_val:,} val rows saved to {out_dir}")
    return summaries, stats

In [15]:
def _test_end_to_end_preprocessing():
    with tempfile.TemporaryDirectory() as processed_dir, tempfile.TemporaryDirectory() as out_dir:
        rng = np.random.default_rng(42)

        # Build 2 synthetic "loaders.py output" files -- 3 units each, 20
        # rows/unit, so a 20% validation split holds out exactly 1 unit
        # (20 rows) per file, leaving 2 units (40 rows) for training.
        specs = {"FAKE_FILE_A.h5": 3, "FAKE_FILE_B.h5": 3}
        for fname, n_units in specs.items():
            rows_per_unit = 20
            n = n_units * rows_per_unit
            unit_col = np.repeat(np.arange(1, n_units + 1), rows_per_unit)
            a = np.stack([unit_col, np.tile(np.arange(rows_per_unit), n_units),
                          np.zeros(n), np.ones(n)], axis=1)
            data = {
                "W": rng.normal(loc=50.0, scale=10.0, size=(n, 4)),
                "X_s": rng.normal(loc=0.0, scale=5.0, size=(n, 14)),
                "X_v": rng.normal(loc=0.0, scale=5.0, size=(n, 14)),
                "A": a,
                "Y": rng.integers(0, 300, size=(n, 1)),
                "labels": np.zeros((n, 5), dtype=np.int8),
            }
            np.savez_compressed(os.path.join(processed_dir, fname.replace(".h5", "_dev.npz")), **data)

        test_files = list(specs.keys())
        summaries, stats = process_and_save_preprocessed(
            processed_dir=processed_dir, out_dir=out_dir, files=test_files, val_fraction=0.2, augment=True)

        # Every file produced a train and a val output on disk.
        assert all(os.path.exists(s["train_path"]) and os.path.exists(s["val_path"]) for s in summaries)

        # No test-split input existed, so no test output should exist either.
        assert all("test_path" not in s for s in summaries)

        # Row accounting: 1 held-out unit x 20 rows = 20 val rows/file;
        # the other 2 units x 20 rows = 40 train rows/file.
        for s in summaries:
            assert s["n_val_rows"] == 20, s
            assert s["n_train_rows"] == 40, s

        # Reload one train file: shapes are right, and W's mean is still
        # near 0 even after augmentation noise was layered on top of
        # normalization (a loose bound -- noise is expected to shift it
        # a little, just not by a lot).
        with np.load(summaries[0]["train_path"]) as npz:
            reloaded_train = {k: npz[k] for k in npz.files}
        assert abs(reloaded_train["W"].mean()) < 1.0
        assert reloaded_train["labels"].shape == (40, 5)

        # Reload one val file: right shape, and -- unlike train -- never augmented.
        with np.load(summaries[0]["val_path"]) as npz:
            reloaded_val = {k: npz[k] for k in npz.files}
        assert reloaded_val["W"].shape == (20, 4)

        print("process_and_save_preprocessed: PASSED (end-to-end, synthetic data, no Drive needed)")


_test_end_to_end_preprocessing()

=== Pass 1: computing per-channel normalization stats (train rows only) ===
[1/2] Accumulating stats from FAKE_FILE_A.h5 ...
    2 train unit(s), 1 validation unit(s), 40 train rows
[2/2] Accumulating stats from FAKE_FILE_B.h5 ...
    2 train unit(s), 1 validation unit(s), 40 train rows
    W: mean=[49.378 49.023 49.441 49.785]  std=[8.439 8.533 9.042 9.474]
    X_s: mean=[-0.668  0.385  0.276  0.445 -0.352 -0.662  0.676 -0.354 -0.055  0.049
  0.39  -0.495  0.628 -0.163]  std=[5.128 4.836 5.519 4.689 5.108 4.914 5.166 4.345 5.475 5.333 5.259 5.32
 4.788 5.38 ]
    X_v: mean=[-0.17  -0.271  1.32   0.737  0.648 -0.473 -0.517 -0.834  0.097  0.044
 -0.424 -0.592 -0.084  0.047]  std=[4.532 4.417 5.25  4.94  5.227 5.185 5.284 5.18  5.294 4.472 4.575 4.631
 5.14  4.869]
Done -- stats computed from 80 total training rows across 2 files

=== Pass 2: normalizing, splitting, saving ===
[1/2] Processing FAKE_FILE_A.h5 ...
    augmented 40 train rows (-6.0dB Gaussian on W, 1.07% impulse on X_s)
   

## 9. Run it

Reads `loaders.py`'s real, Drive-saved dev-split output, computes real
normalization statistics, augments, splits, and saves the real
train/validation `.npz` files.

**This cell needs Drive mounted and `loaders.py` already run** (its
output present at `DEFAULT_PROCESSED_DIR`) -- skip it and rely on
Section 8's self-contained test above if you are only reviewing the
logic without Drive access.

In [16]:
summaries, stats = process_and_save_preprocessed()
summaries

=== Pass 1: computing per-channel normalization stats (train rows only) ===
[1/9] Accumulating stats from N-CMAPSS_DS01-005.h5 ...
    5 train unit(s), 1 validation unit(s), 4,069,786 train rows
[2/9] Accumulating stats from N-CMAPSS_DS02-006.h5 ...
    5 train unit(s), 1 validation unit(s), 4,495,287 train rows
[3/9] Accumulating stats from N-CMAPSS_DS03-012.h5 ...
    7 train unit(s), 2 validation unit(s), 4,132,089 train rows
[4/9] Accumulating stats from N-CMAPSS_DS04.h5 ...
    5 train unit(s), 1 validation unit(s), 5,190,386 train rows
[5/9] Accumulating stats from N-CMAPSS_DS05.h5 ...
    5 train unit(s), 1 validation unit(s), 3,376,602 train rows
[6/9] Accumulating stats from N-CMAPSS_DS06.h5 ...
    5 train unit(s), 1 validation unit(s), 3,272,497 train rows
[7/9] Accumulating stats from N-CMAPSS_DS07.h5 ...
    5 train unit(s), 1 validation unit(s), 3,285,447 train rows
[8/9] Accumulating stats from N-CMAPSS_DS08a-009.h5 ...
    7 train unit(s), 2 validation unit(s), 3,911,36

[{'filename': 'N-CMAPSS_DS01-005.h5',
  'n_train_rows': 4069786,
  'n_val_rows': 836850,
  'train_path': '/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/preprocessed/N-CMAPSS_DS01-005_train.npz',
  'val_path': '/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/preprocessed/N-CMAPSS_DS01-005_val.npz'},
 {'filename': 'N-CMAPSS_DS02-006.h5',
  'n_train_rows': 4495287,
  'n_val_rows': 768160,
  'train_path': '/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/preprocessed/N-CMAPSS_DS02-006_train.npz',
  'val_path': '/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/preprocessed/N-CMAPSS_DS02-006_val.npz'},
 {'filename': 'N-CMAPSS_DS03-012.h5',
  'n_train_rows': 4132089,
  'n_val_rows': 1439188,
  'train_path': '/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/preprocessed/N-CMAPSS_DS03-012_train.npz',
  'val_path': '/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/preprocessed/N-CMAPSS_DS03-012_val.npz'},
 {'filena

In [19]:
# Known dev-unit counts per file, taken directly from docs/data-dictionary.md
# -- this is the "ground truth" input to the split logic, not something the
# pipeline computed itself.
dev_units = {"DS01": 6, "DS02": 6, "DS03": 9, "DS04": 6, "DS05": 6,
             "DS06": 6, "DS07": 6, "DS08a": 9, "DS08c": 6}

# The validation-unit counts process_and_save_preprocessed() actually
# printed during the real Colab run -- copied here by hand from that
# output, not re-read from any file.
printed_val_units = {"DS01": 1, "DS02": 1, "DS03": 2, "DS04": 1, "DS05": 1,
                      "DS06": 1, "DS07": 1, "DS08a": 2, "DS08c": 1}

# For each file, recompute what split_units_train_val() SHOULD have
# produced -- round(0.2 * n_units) -- using the exact same formula the
# function itself uses (VALIDATION_UNIT_FRACTION = 0.2), then compare
# that expected value against what was actually printed.
for name, n in dev_units.items():
    expected = round(0.2 * n)          # same formula as split_units_train_val()
    actual = printed_val_units[name]   # what the real run actually reported
    status = "OK" if expected == actual else "MISMATCH"  # flag any disagreement loudly
    print(f"{name}: {n} units -> expected {expected} val, got {actual}  [{status}]")

DS01: 6 units -> expected 1 val, got 1  [OK]
DS02: 6 units -> expected 1 val, got 1  [OK]
DS03: 9 units -> expected 2 val, got 2  [OK]
DS04: 6 units -> expected 1 val, got 1  [OK]
DS05: 6 units -> expected 1 val, got 1  [OK]
DS06: 6 units -> expected 1 val, got 1  [OK]
DS07: 6 units -> expected 1 val, got 1  [OK]
DS08a: 9 units -> expected 2 val, got 2  [OK]
DS08c: 6 units -> expected 1 val, got 1  [OK]


## What's next

With `preprocessing.py` producing normalized, augmented, unit-split
train/validation `.npz` files (and test files, once `loaders.py` is run
with `split="test"`), the next real piece of work is
`models/ma1dcnn.py` -- the actual Multi-Head Attention 1D-CNN
architecture, per Wang et al. (2020) -- followed by
`training/train.py` to train it on this preprocessed data. See
`deliverables/next_steps_roadmap.docx` for the full six-phase plan.